# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Clustering.

My question is "what kinds of pages exist across the content inventory?" — per the framing skill's mapping table, that phrasing ("what kinds of items exist?") maps directly to clustering, not classification (no predefined label to sort into), not ranking/scoring (I'm not producing one ordered priority list), and not signal analysis (I'm not testing which individual signals correlate with an outcome). I'm grouping pages by similarity across several metrics at once  position, engagement, freshness, word count, volume  to find structure, not to predict a known answer.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

There is no target label in the supervised sense  clustering is unsupervised, so nothing is being predicted from a known answer. What I define instead is the feature set the clusters get built from (avg_position, engagement_rate, word_count, impressions_90d, content_age_days, etc.). The cluster assignment that comes out afterward is a derived grouping, not an observed outcome and not a hand-written rule  it's an output of the algorithm's own similarity logic, which I then interpret and name.

One explicit exclusion, per the data skill's label-trap warning: trend_direction and trend_pct are never used as clustering features, since trend_direction is itself computed from trend_pct  including either would let a pre-existing rule quietly shape the grouping instead of letting the raw signals speak for themselves.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

Silhouette score, paired with a manual sanity check.

Silhouette score measures, for each page, whether it sits closer to its own cluster's center or to a neighboring cluster's center, averaged across all pages, on a scale from -1 to 1. A score meaningfully above 0 I'll use >0.25 as my working "good" threshold, to be calibrated once I see the real distribution  means the clusters are genuinely separated groups rather than an arbitrary slice through one continuous blob.

Silhouette score alone can be gamed by a degenerate clustering (e.g. one giant cluster plus a few tiny outlier clusters still scores deceptively well), so I'll pair it with pulling 5–10 real pages from each cluster and confirming by eye that they intuitively belong together before trusting the number

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one content page (content_id). Grain is verified with a probe below, not assumed — per the data skill's rule, a blind drop_duplicates can silently discard meaningful rows if content_id isn't actually unique.

In [13]:
import pandas as pd

# Utiliser le DataFrame déjà chargé depuis GitHub
raw = df_from_github.copy()
print("Raw rows:", len(raw))

# Grain probe — confirm one row per content_id, don't assume it
grain_check = raw.groupby("content_id").size()
violations = grain_check[grain_check > 1]
print("content_ids with more than 1 row:", len(violations))
if len(violations) > 0:
    print(violations.head())

# Apply the lane guide's filter rule
df = raw[(raw["impressions_90d"] > 0) & (raw["content_age_days"] >= 90)].copy()
df_position = df[df["avg_position"] > 0]  # avg_position == 0 means "no data," not rank zero

print("\nRows after impressions/age filter:", len(df))

# Show the actual unit of analysis — one row = one page's metric profile
df[["content_id", "avg_position", "engagement_rate", "word_count", "impressions_90d"]].head()

Raw rows: 30000
content_ids with more than 1 row: 0

Rows after impressions/age filter: 30000


,content_id,avg_position,engagement_rate,word_count,impressions_90d
0,content_304f48230142,10.6,5.88,3221.0,3803
1,content_a1fb4e703a9e,20.3,0.00,2481.0,15320
2,content_9aa793d4d895,36.5,0.00,3515.0,12581
3,content_331d6c4de07b,6.2,1.28,NaN,11751
4,content_d99b7a2d90ca,44.0,0.00,2803.0,19140


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed if-statement rule requires deciding in advance which combinations of conditions define a group — e.g. "if position < 10 AND engagement < 0.3%, call it archetype X." That works when one or two dimensions matter, but page archetypes here likely depend on interactions across many metrics simultaneously: position, engagement, freshness, word count, and impression volume. A page can be low-position but high-engagement, or high-impressions but stale — the useful groupings aren't obvious ahead of time, and an if-statement chain would need dozens of hand-guessed nested branches to capture them, with someone guessing the right thresholds for every combination.

Clustering instead lets the data find which pages actually sit close together across all dimensions at once, surfacing groupings nobody would have thought to hard-code in advance — that's the concrete justification for ML over a rulebook here, not just "ML sounds more advanced."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

### 1. Monter Google Drive

Pour monter votre Google Drive, exécutez la cellule de code ci-dessous. Vous devrez suivre un lien et autoriser Google Colab à accéder à votre Drive.

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### 2. Accéder aux fichiers sur Google Drive

Une fois votre Drive monté, vos fichiers seront accessibles sous `/content/drive/My Drive/`. Par exemple, si votre fichier `content_refresh_anonymized.csv` se trouve dans un dossier nommé `data/raw` dans votre Google Drive, le chemin serait `/content/drive/My Drive/data/raw/content_refresh_anonymized.csv`.

Voici un exemple pour charger ce fichier dans un DataFrame pandas:

In [12]:
import pandas as pd

# Remplacez cette URL par l'URL raw (brute) de votre fichier CSV sur GitHub
github_csv_url = 'https://raw.githubusercontent.com/Beni242/FlyRank_intership/main/data/raw/content_refresh_anonymized.csv'

try:
    df_from_github = pd.read_csv(github_csv_url)
    print(f"Fichier '{github_csv_url}' chargé avec succès.")
    display(df_from_github.head())
except Exception as e:
    print(f"Une erreur est survenue lors du chargement du fichier depuis GitHub : {e}")

Fichier 'https://raw.githubusercontent.com/Beni242/FlyRank_intership/main/data/raw/content_refresh_anonymized.csv' chargé avec succès.


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7
